# Benchmark: PRS scoring speed

Times `calculate_prs` on a real *All of Us* VDS window, so a scoring-path change
can be checked for a speed regression before it merges.

**This is not a validation tier** -- it asserts nothing about correctness (the
integration tests and `validate_scoring_on_aou.ipynb` do that). It only measures
wall-clock time.

## How to compare two versions

Run this notebook once per install and compare the reported medians. Install one
version, **restart the kernel**, run top-to-bottom, note the median; then repeat
for the other version:

    !pip install -q "git+https://github.com/dokyoonkimlab/aoutools.git@v0.1.2"      # baseline
    !pip install -q "git+https://github.com/dokyoonkimlab/aoutools.git@optmization"  # candidate

The setup cell prints the version it actually measured. Keep `N_SAMPLES`,
`PGS_ID`, and `chunk_size` identical between the two runs, or it is not an
apples-to-apples comparison.

Also watch Hail's per-chunk progress bars: the change this benchmark exists for
turns two full `1->1000` passes per chunk (the split-and-join ran twice) into
one heavy per-sample pass plus one cheap rows-only pass.

In [ ]:
# !pip install -q "git+https://github.com/dokyoonkimlab/aoutools.git@optmization"

import logging
import pathlib
import statistics
import time

import hail as hl

import aoutools
from aoutools import get_vds_path, get_workspace_bucket, init_hail
from aoutools.prs import (
    PRSConfig,
    calculate_prs,
    download_pgs,
    read_prs_weights,
)

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s  %(levelname)-7s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logging.getLogger("aoutools").setLevel(logging.INFO)

init_hail()
print("aoutools version:", aoutools.__version__)

VDS_PATH = get_vds_path()
BUCKET = get_workspace_bucket()
OUT = f"{BUCKET}/prs_benchmark"

# --- knobs: keep these identical across the versions you compare ---
N_SAMPLES = 2000  # cohort size; raise toward the full VDS for a real timing
PGS_ID = "PGS000746"  # 163 variants, all matched (see measure_minrep notebook)
REPEATS = 3  # timed runs; the first is a warm-up, reported separately
CONFIG = PRSConfig(chunk_size=20000, include_n_matched=True)

In [ ]:
vds = hl.vds.read_vds(VDS_PATH)

sample_ids = vds.variant_data.cols().head(N_SAMPLES).s.collect()
samples_ht = hl.Table.parallelize(
    [{"s": s} for s in sample_ids], hl.tstruct(s=hl.tstr)
).key_by("s")
vds = hl.vds.filter_samples(vds, samples_ht)
print(f"{len(sample_ids)} samples")

In [ ]:
pgs_dir = pathlib.Path.home() / "pgs_downloads"
pgs_dir.mkdir(parents=True, exist_ok=True)
download_pgs(
    outdir=str(pgs_dir),
    pgs=[PGS_ID],
    build="GRCh38",
    overwrite_existing_file=True,
)
weights_file = sorted(pgs_dir.rglob(f"{PGS_ID}*.txt.gz"))[0]

PGS_COLUMN_MAP = {
    "chr": "hm_chr",
    "pos": "hm_pos",
    "effect_allele": "effect_allele",
    "noneffect_allele": "other_allele",
    "weight": "effect_weight",
}
weights = read_prs_weights(
    file_path=str(weights_file),  # local path -> staged to GCS for us
    header=True,
    column_map=PGS_COLUMN_MAP,
    delimiter="\t",
    comment="#",
    validate_alleles=True,
    missing="",
    force=True,
)
print(f"{PGS_ID}: {weights.count():,} variants")

In [ ]:
# Quiet the per-chunk INFO timings so the loop output stays readable; the wall
# clock below is what we compare. Raise aoutools back to INFO to watch phases.
logging.getLogger("aoutools").setLevel(logging.WARNING)

timings = []
for i in range(REPEATS):
    start = time.perf_counter()
    calculate_prs(
        weights_table=weights,
        vds=vds,
        output_path=f"{OUT}/run_{i}.csv",
        config=CONFIG,
    )
    elapsed = time.perf_counter() - start
    timings.append(elapsed)
    tag = "warm-up" if i == 0 else "timed  "
    print(f"run {i}  [{tag}]  {elapsed:8.1f} s")

steady = timings[1:] or timings
print(
    f"\naoutools {aoutools.__version__}  |  {len(sample_ids)} samples  |  "
    f"{PGS_ID}\n"
    f"median of {len(steady)} timed run(s): {statistics.median(steady):.1f} s"
)

## Reading the result

* Compare the **median timed run** across the two installs. The first run is a
  warm-up (JVM/shuffle) and is reported separately.
* Watch the per-chunk progress bars while it runs. Before the fix, each chunk
  showed two full `1->1000` passes; after, one heavy per-sample pass plus a fast
  rows-only pass.
* If the candidate is not faster, the rows-only pass is not cheap on this VDS
  (Hail did not prune entries from `split_multi`); the fallback is to
  `checkpoint` the split MatrixTable once per chunk instead.